# 🔍 Polars Pre-Load Data Validation
### Sample dataset + all 4 validation scripts

This notebook includes:
1. **Dataset generation** — creates `sample_data.csv` with intentional transient & non-transient errors
2. **Script 1** — Schema & Type Validation
3. **Script 2** — Null / Missing Value Scanner  
4. **Script 3** — Value Range & Constraint Scanner
5. **Script 4** — Combined Pre-Load Report

---


## Setup — Install Polars

In [ ]:
!pip install polars

---
## 📦 Generate Sample Dataset

You can download the sample_data.csv file or run the code to create `sample_data.csv` (504 rows) with **intentionally injected errors**:

| Error | Rows | Type |
|-------|------|------|
| Missing `age` (nulls) | 11 rows | Transient |
| Bad `price` string (`N/A`) | 3 rows | Transient |
| Invalid `status` value | 2 rows | Transient |
| Large null block in `score` | ~60 rows (~12%) | **Non-transient** |
| Duplicate rows | 4 rows | Transient |


In [ ]:
import csv
import random
from datetime import datetime, timedelta

random.seed(42)

statuses = ["active", "inactive", "pending"]
names = ["Alice","Bob","Carol","Dave","Eve","Frank","Grace","Heidi","Ivan","Judy",
         "Karl","Laura","Mallory","Niaj","Olivia","Peggy","Rupert","Sybil","Trent","Victor"]

rows = []
for i in range(1, 501):
    row = {
        "id": i,
        "name": random.choice(names) + f"_{i}",
        "age": random.randint(18, 80),
        "price": round(random.uniform(5.0, 999.0), 2),
        "status": random.choice(statuses),
        "date": (datetime(2024,1,1) + timedelta(days=random.randint(0,364))).strftime("%Y-%m-%d"),
        "score": round(random.uniform(0, 100), 1),
    }
    # Inject transient errors (~3%)
    if i in [7, 23, 45, 88, 102, 167, 201, 289, 310, 412, 467]:
        row["age"] = ""           # null / missing age
    if i in [15, 78, 234]:
        row["price"] = "N/A"      # bad price string
    if i in [55, 199]:
        row["status"] = "unknown" # invalid status value

    # Inject non-transient errors (structural block)
    if 350 <= i <= 410:
        row["score"] = ""         # large null block in score (~12%)
    if i in [100, 200, 300, 400]:
        rows.append(row)          # duplicate rows

    rows.append(row)

with open("sample_data.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id","name","age","price","status","date","score"])
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ Generated sample_data.csv with {len(rows)} rows")
print("   Columns: id, name, age, price, status, date, score")


### Preview the dataset

In [1]:
import polars as pl
pl.read_csv("sample_data.csv", infer_schema_length=0).head(10)


id,name,age,price,status,date,score
str,str,str,str,str,str,str
"""1""","""Dave_1""","""19""","""742.1""","""active""","""2024-04-24""","""14.0"""
"""2""","""Dave_2""","""61""","""741.22""","""pending""","""2024-02-14""","""59.0"""
"""3""","""Bob_3""","""19""","""98.13""","""active""","""2024-09-15""","""60.2"""
"""4""","""Sybil_4""","""30""","""716.72""","""pending""","""2024-10-06""","""42.0"""
"""5""","""Olivia_5""","""55""","""281.52""","""active""","""2024-03-22""","""69.8"""
"""6""","""Karl_6""","""35""","""159.55""","""inactive""","""2024-02-22""","""9.3"""
"""7""","""Dave_7""",null,"""847.41""","""pending""","""2024-05-15""","""80.7"""
"""8""","""Olivia_8""","""52""","""129.08""","""inactive""","""2024-02-10""","""55.2"""
"""9""","""Victor_9""","""74""","""861.54""","""pending""","""2024-04-08""","""70.5"""


---
## Script 1 — Schema & Type Validation

Checks that all expected columns exist and have the correct data types.  
A **missing column** is always non-transient. A **type mismatch** may be transient if 
Polars inferred a broader type due to dirty values.


In [3]:
import polars as pl

def scan_schema(filepath: str, expected_schema: dict):
    lf = pl.scan_csv(filepath)
    actual = lf.collect_schema()

    print("=== Schema Validation ===")
    for col, expected_type in expected_schema.items():
        if col not in actual:
            print(f"[NON-TRANSIENT] Missing column: '{col}'")
        elif actual[col] != expected_type:
            print(f"[TRANSIENT?]    Column '{col}': expected {expected_type}, got {actual[col]}")
        else:
            print(f"[OK]            Column '{col}' matches {expected_type}")

expected = {
    "id":     pl.Int64,
    "name":   pl.String,
    "age":    pl.Int64,
    "price":  pl.Float64,
    "status": pl.String,
    "date":   pl.String,
    "score":  pl.Float64,
}

scan_schema("sample_data.csv", expected)


=== Schema Validation ===
[OK]            Column 'id' matches Int64
[OK]            Column 'name' matches String
[OK]            Column 'age' matches Int64
[TRANSIENT?]    Column 'price': expected Float64, got String
[OK]            Column 'status' matches String
[OK]            Column 'date' matches String
[OK]            Column 'score' matches Float64


---
## Script 2 — Null / Missing Value Scanner

Scans every column for nulls. Uses lazy evaluation so the full file is never loaded into RAM.

- **< 5% null rate** → *Transient* (e.g., a few feed dropouts)  
- **≥ 5% null rate** → *Non-transient* (structural upstream problem)


In [4]:
import polars as pl

def scan_nulls(filepath: str, null_threshold: float = 0.05):
    lf = pl.scan_csv(filepath)
    total_rows = lf.select(pl.len()).collect().item()

    null_counts = (
        lf.select(pl.all().is_null().sum())
        .collect()
        .transpose(include_header=True, column_names=["null_count"])
    )

    print(f"=== Null Scan (total rows: {total_rows}) ===")
    for row in null_counts.iter_rows(named=True):
        col   = row["column"]
        count = row["null_count"]
        rate  = count / total_rows

        if count == 0:
            print(f"[OK]            '{col}' — no nulls")
        elif rate < null_threshold:
            print(f"[TRANSIENT]     '{col}' — {count} nulls ({rate:.1%}) — may self-correct")
        else:
            print(f"[NON-TRANSIENT] '{col}' — {count} nulls ({rate:.1%}) — structural issue")

scan_nulls("sample_data.csv", null_threshold=0.05)


=== Null Scan (total rows: 504) ===
[OK]            'id' — no nulls
[OK]            'name' — no nulls
[TRANSIENT]     'age' — 11 nulls (2.2%) — may self-correct
[OK]            'price' — no nulls
[OK]            'status' — no nulls
[OK]            'date' — no nulls
[NON-TRANSIENT] 'score' — 62 nulls (12.3%) — structural issue


---
## Script 3 — Value Range & Constraint Scanner

Validates column values against business rules:
- **Numeric range** — e.g. age must be 0–120
- **Allowed values** — e.g. status must be one of `active / inactive / pending`

Thresholds:
- **< 10 violations** → *Transient*
- **≥ 10 violations** → *Non-transient*


In [4]:
import polars as pl

def scan_constraints(filepath: str, constraints: dict):
    lf = pl.scan_csv(filepath, infer_schema_length=0)  # ← critical

    print("=== Constraint Scan ===")
    for col, rule in constraints.items():
        if isinstance(rule, tuple):
            lo, hi = rule
            bad = (
                lf.select(
                    pl.col(col).cast(pl.Float64, strict=False)  # ← cast to numeric
                )
                .filter(
                    pl.col(col).is_not_null() &
                    ((pl.col(col) < lo) | (pl.col(col) > hi))
                )
                .select(pl.len())
                .collect()
                .item()
            )
            tag = "[TRANSIENT]    " if bad < 10 else "[NON-TRANSIENT]"
            status = "✅ none" if bad == 0 else f"⚠️  {bad} rows"
            print(f"{tag} '{col}' out of range [{lo}, {hi}]: {status}")

        elif isinstance(rule, list):
            bad = (
                lf.select(
                    pl.col(col).str.strip_chars()
                )
                .filter(
                    pl.col(col).is_not_null() &
                    ~pl.col(col).is_in(rule)
                )
                .select(pl.len())
                .collect()
                .item()
            )
            tag = "[TRANSIENT]    " if bad < 10 else "[NON-TRANSIENT]"
            status = "✅ none" if bad == 0 else f"⚠️  {bad} rows"
            print(f"{tag} '{col}' invalid values (allowed: {rule}): {status}")

constraints = {
    "age":    (0, 120),
    "price":  (0.0, 99999.0),
    "score":  (0.0, 100.0),
    "status": ["active", "inactive", "pending"],
}

scan_constraints("sample_data.csv", constraints)


=== Constraint Scan ===
[TRANSIENT]     'age' out of range [0, 120]: ✅ none
[TRANSIENT]     'price' out of range [0.0, 99999.0]: ✅ none
[TRANSIENT]     'score' out of range [0.0, 100.0]: ✅ none
[TRANSIENT]     'status' invalid values (allowed: ['active', 'inactive', 'pending']): ⚠️  2 rows


```

### Quick checklist before re-running code above:

1. **Restart the kernel** (`Kernel → Restart & Run All`) — stale function definitions from earlier cells can persist in memory and cause the old version to run even after editing
2. Confirm the cell you edited is the one actually executing (Jupyter can have duplicate function definitions across cells if you copied code around)

The expected output after a clean run:
```
=== Constraint Scan ===
[TRANSIENT]     'age' out of range [0, 120]: ✅ none
[TRANSIENT]     'price' out of range [0.0, 99999.0]: ✅ none
[TRANSIENT]     'score' out of range [0.0, 100.0]: ✅ none
[TRANSIENT]     'status' invalid values (allowed: ['active', 'inactive', 'pending']): ⚠️  2 rows

In [11]:
import polars as pl

# Checking actual raw values in the status column to confirm whitespace or other errors
lf = pl.scan_csv("sample_data.csv", infer_schema_length=0)

print(lf.select(pl.col("status").unique()).collect())
print()
# Show exact bytes to catch hidden whitespace/encoding issues
print(lf.select(pl.col("status").unique())
        .collect()
        .to_series()
        .to_list())

shape: (4, 1)
┌──────────┐
│ status   │
│ ---      │
│ str      │
╞══════════╡
│ pending  │
│ active   │
│ inactive │
│ unknown  │
└──────────┘

['active', 'unknown', 'pending', 'inactive']


---
## Script 4 — Combined Pre-Load Report

Runs all checks in one pass and prints a structured health summary.  
Uses `.fetch(1000)` to sample 1000 rows for speed on very large files.


In [3]:
import polars as pl
from datetime import datetime

def pre_load_report(filepath: str):
    lf        = pl.scan_csv(filepath)
    df_sample = lf.collect().head(1000)
    total     = lf.select(pl.len()).collect().item()

    print(f"\n{'='*55}")
    print(f"  Pre-Load Report: {filepath}")
    print(f"  Timestamp : {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"  Total rows: {total:,}  |  Columns: {df_sample.width}")
    print(f"{'='*55}")

    # --- Nulls ---
    print("\n[NULLS]")
    found_nulls = False
    for col in df_sample.columns:
        n    = df_sample[col].null_count()
        rate = n / len(df_sample)
        if n > 0:
            found_nulls = True
            kind = "NON-TRANSIENT" if rate >= 0.05 else "TRANSIENT    "
            print(f"  [{kind}] '{col}': {n} nulls in sample ({rate:.1%})")
    if not found_nulls:
        print("  [OK] No nulls found in sample")

    # --- Duplicates ---
    print("\n[DUPLICATES]")
    dupes = len(df_sample) - df_sample.n_unique()
    if dupes:
        kind = "NON-TRANSIENT" if dupes >= 10 else "TRANSIENT    "
        print(f"  [{kind}] {dupes} duplicate rows in sample")
    else:
        print("  [OK] No duplicates in sample")

    # --- Type anomalies ---
    print("\n[TYPE ANOMALIES]")
    found_anomalies = False
    for col in df_sample.columns:
        if df_sample[col].dtype == pl.String:
            coerced       = df_sample[col].cast(pl.Float64, strict=False).is_null().sum()
            original_null = df_sample[col].is_null().sum()
            new_nulls     = coerced - original_null
            if new_nulls > 0:
                found_anomalies = True
                kind = "NON-TRANSIENT" if new_nulls >= 10 else "TRANSIENT    "
                print(f"  [{kind}] '{col}': {new_nulls} values cannot cast to Float64")
    if not found_anomalies:
        print("  [OK] No type anomalies detected")

    print(f"\n{'='*55}\n")

pre_load_report("sample_data.csv")



  Pre-Load Report: sample_data.csv
  Timestamp : 2026-03-26 07:33:45
  Total rows: 504  |  Columns: 7

[NULLS]
  [TRANSIENT    ] 'age': 11 nulls in sample (2.2%)
  [NON-TRANSIENT] 'score': 62 nulls in sample (12.3%)

[DUPLICATES]
  [TRANSIENT    ] 4 duplicate rows in sample

[TYPE ANOMALIES]
  [NON-TRANSIENT] 'name': 504 values cannot cast to Float64
  [TRANSIENT    ] 'price': 3 values cannot cast to Float64
  [NON-TRANSIENT] 'status': 504 values cannot cast to Float64
  [NON-TRANSIENT] 'date': 504 values cannot cast to Float64




---
## 📋 Error Classification Reference

| Error | Rows Affected | Classification |
|-------|--------------|----------------|
| Missing `age` | 11 rows | Transient |
| Bad `price` string (`N/A`) | 3 rows | Transient |
| Invalid `status` value | 2 rows | Transient |
| Duplicate rows | 4 rows | Transient |
| Large null block in `score` | ~61 rows (~12%) | **Non-transient** |

> **Transient** errors may self-correct on retry (feed blips, timing issues).  
> **Non-transient** errors require upstream investigation before loading.
